In [27]:
# Import needed libraries
import pandas as pd
from sqlalchemy import create_engine

# Connect to the db we created
Engine = create_engine("sqlite:///..//data/processed/football_scouting.db")

# Read the whole Match_Metrics table into a DataFrame (the "league" for comparisons)
league_df = pd.read_sql("SELECT * FROM Match_Metrics", Engine)

print("Rows and Columns: ", league_df.shape)
league_df.head()

# MISSING DATA

Rows and Columns:  (1787, 22)


,match_id,player_id,player_name,position,total_passes,total_completed_passes,total_accuracy_passes,total_progressive_passes,total_long_passes,total_won_duels,...,total_recoveries,total_tackles_won,total_shots,total_goals,total_assists,total_xg,total_dribbles,total_completed_dribbles,minutes_played,rating
0,8650,2954,Youri Tielemans,Left Center Midfield,1,1,None,0,0,0,...,0,0,0,0,0,0.000,0,0,9,None
1,8650,3054,Fernando Luiz Roza,Center Midfield,66,57,None,9,5,0,...,7,0,1,0,0,0.066,0,0,95,None
2,8650,3077,Jan Vertonghen,Left Back,47,40,None,8,8,0,...,2,0,0,0,0,0.000,1,1,95,None
3,8650,3089,Kevin De Bruyne,Secondary Striker,42,33,None,13,8,1,...,10,1,3,1,0,0.070,2,2,95,None
4,8650,3101,Vincent Kompany,Left Center Back,44,41,None,8,14,1,...,1,1,1,0,0,0.057,0,0,95,None


In [28]:
# How many empty values do we have?
print(league_df.isna().sum())

# Basic Stats of every numeric column
league_df.describe()

match_id                             0
player_id                            0
player_name                          0
position                             0
total_passes                         0
total_completed_passes               0
total_accuracy_passes             1787
total_progressive_passes             0
total_long_passes                    0
total_won_duels                      0
total_duels                          0
total_successful_interceptions       0
total_recoveries                     0
total_tackles_won                    0
total_shots                          0
total_goals                          0
total_assists                        0
total_xg                             0
total_dribbles                       0
total_completed_dribbles             0
minutes_played                       0
rating                            1787
dtype: int64


,match_id,player_id,total_passes,total_completed_passes,total_progressive_passes,total_long_passes,total_won_duels,total_duels,total_successful_interceptions,total_recoveries,total_tackles_won,total_shots,total_goals,total_assists,total_xg,total_dribbles,total_completed_dribbles,minutes_played
count,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000,1787.000000
mean,7694.250140,5224.258534,35.188025,28.073307,8.221600,6.129267,0.644656,1.757694,0.456631,3.177392,0.644656,0.932848,0.087857,0.053721,0.096669,1.180190,0.733072,76.595411
std,364.523487,1879.600538,24.716702,22.262852,7.060856,5.956878,0.951028,1.789305,0.809860,2.492179,0.951028,1.283006,0.313209,0.230442,0.208805,1.866781,1.329667,30.288298
min,7525.000000,2941.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000
25%,7543.500000,4417.000000,17.000000,11.000000,3.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,62.000000
50%,7560.000000,5476.000000,31.000000,23.000000,7.000000,5.000000,0.000000,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,94.000000
75%,7579.000000,5620.000000,49.000000,40.000000,12.000000,9.000000,1.000000,3.000000,1.000000,5.000000,1.000000,1.000000,0.000000,0.000000,0.101500,2.000000,1.000000,95.000000
max,8658.000000,20016.000000,180.000000,168.000000,58.000000,40.000000,5.000000,10.000000,7.000000,15.000000,5.000000,11.000000,3.000000,2.000000,2.221000,18.000000,15.000000,128.000000


In [29]:
# How many rows do we have for each position?
league_df['position'].value_counts()

position
Center Forward               140
Left Center Back             138
Right Center Back            132
Goalkeeper                   129
Left Center Midfield         123
Left Back                    123
Right Back                   120
Right Center Midfield        120
Right Wing                   110
Left Wing                    109
Left Midfield                 76
Center Attacking Midfield     73
Right Midfield                72
Left Center Forward           56
Right Center Forward          55
Center Defensive Midfield     51
Right Defensive Midfield      36
Left Defensive Midfield       36
Center Back                   22
Right Wing Back               14
Left Wing Back                13
Secondary Striker             11
Right Attacking Midfield      11
Left Attacking Midfield       10
Center Midfield                7
Name: count, dtype: int64

In [30]:
# Match detailed position to a broad group (Midfilder, Defender, Forward)
def get_position_group(position):
    if position is None:
        return None
    if position == "Goalkeeper":
        return "Goalkeeper"
    if "Back" in position:
        return "Defender"
    if "Attacking" in position:     # Attacking Midfielders
        return "Forward"
    if "Midfield" in position:
        return "Midfielder"
    return "Forward"

league_df["position_group"] = league_df["position"].apply(get_position_group)
league_df["position_group"].value_counts()

position_group
Forward       575
Defender      562
Midfielder    521
Goalkeeper    129
Name: count, dtype: int64

In [31]:
league_df

,match_id,player_id,player_name,position,total_passes,total_completed_passes,total_accuracy_passes,total_progressive_passes,total_long_passes,total_won_duels,...,total_tackles_won,total_shots,total_goals,total_assists,total_xg,total_dribbles,total_completed_dribbles,minutes_played,rating,position_group
0,8650,2954,Youri Tielemans,Left Center Midfield,1,1,None,0,0,0,...,0,0,0,0,0.000,0,0,9,None,Midfielder
1,8650,3054,Fernando Luiz Roza,Center Midfield,66,57,None,9,5,0,...,0,1,0,0,0.066,0,0,95,None,Midfielder
2,8650,3077,Jan Vertonghen,Left Back,47,40,None,8,8,0,...,0,0,0,0,0.000,1,1,95,None,Defender
3,8650,3089,Kevin De Bruyne,Secondary Striker,42,33,None,13,8,1,...,1,3,1,0,0.070,2,2,95,None,Forward
4,8650,3101,Vincent Kompany,Left Center Back,44,41,None,8,14,1,...,1,1,0,0,0.057,0,0,95,None,Defender
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1782,7559,5263,Mohamed Abdul Shafy Sayed Abouezeid,Left Back,45,38,None,10,8,0,...,0,0,0,0,0.000,1,1,95,None,Defender
1783,7559,5716,Hussain Al Mogahwi,Right Center Midfield,72,68,None,9,6,1,...,1,4,0,0,0.169,2,1,95,None,Midfielder
1784,7559,6290,Motaz Ali Hassan Hawsawi,Left Center Back,31,27,None,4,3,2,...,2,0,0,0,0.000,0,0,95,None,Defender
1785,7559,6291,Yasser Abdullah Al Mosailem,Goalkeeper,13,12,None,5,9,0,...,0,0,0,0,0.000,0,0,95,None,Goalkeeper


In [32]:
# Import needed libraries
import plotly.express as px

fig = px.box(
    league_df,
    x = "position_group",
    y = "total_progressive_passes",
    title = "Progressive passes per player per match, by position gorup!"
)

fig.show()

In [33]:
print(league_df["minutes_played"].describe())
print("Missing: ", league_df["minutes_played"].isna().sum())

# Sum of minutes per match: about 2 teams * 11 players x 90-95 min -> 2000ish
league_df.groupby("match_id")["minutes_played"].sum().head()

count    1787.000000
mean       76.595411
std        30.288298
min         2.000000
25%        62.000000
50%        94.000000
75%        95.000000
max       128.000000
Name: minutes_played, dtype: float64
Missing:  0


match_id
7525    2068
7529    2068
7530    2090
7531    2090
7532    2090
Name: minutes_played, dtype: int64

In [34]:
## ONE ROW PER PLAYER

# Import needed libraries
import pandas as pd
import scipy.stats as stats     # HUH?

MIN_MINUTES = 45    # ignore appearances shorter than this    WHY?

# Keep only appearances with enough minutes
df = league_df[league_df["minutes_played"] >= MIN_MINUTES].copy()

# One row per player: add up all their matches
players_df = df.groupby(["player_id", "player_name"]).agg(
    position_group = ("position_group", lambda s: s.mode().iloc[0]),    # most frequent group
    matches = ("match_id", "count"),
    minutes = ("minutes_played", "sum"),
    passes = ("total_passes", "sum"),
    completed = ("total_completed_passes", "sum"),
    progressive = ("total_progressive_passes", "sum"),
    duels_won = ("total_won_duels", "sum"),
    duels = ("total_duels", "sum"),
    interceptions = ("total_successful_interceptions", "sum"),
    recoveries = ("total_recoveries", "sum"),
    tackles_won = ("total_tackles_won", "sum"),
    shots = ("total_shots", "sum"),
    goals = ("total_goals", "sum"),
    assists = ("total_assists", "sum"),
    xg = ("total_xg", "sum"),
    dribbles = ("total_dribbles", "sum"),
    completed_dribbles = ("total_completed_dribbles", "sum"),
).reset_index()

# Metrics per 90 min (fair for players with different minutes)
players_df["passes_per90"] = players_df["passes"] / players_df["minutes"] * 90
players_df["progressive_passes_per90"] = players_df["progressive"] / players_df["minutes"] * 90
players_df["interceptions_per90"] = players_df["interceptions"] / players_df["minutes"] * 90
players_df["recoveries_per90"] = players_df["recoveries"] / players_df["minutes"] * 90
players_df["tackles_won_per90"] = players_df["tackles_won"] / players_df["minutes"] * 90
players_df["shots_per90"] = players_df["shots"] / players_df["minutes"] * 90
players_df["xg_per90"] = players_df["xg"] / players_df["minutes"] * 90
players_df["goals_per90"] = players_df["goals"] / players_df["minutes"] * 90
players_df["assists_per90"] = players_df["assists"] / players_df["minutes"] * 90
players_df["dribbles_per90"] = players_df["dribbles"] / players_df["minutes"] * 90

# Percentages (NOT PER 90)
players_df["pass_accuracy_pct"] = players_df["completed"] / players_df["passes"] * 100
players_df["duels_won_pct"] = players_df["duels_won"] / players_df["duels"] * 100
players_df["dribble_success_pct"] = players_df["completed_dribbles"] / players_df["dribbles"] * 100

print("Players: ", players_df.shape[0])
print(players_df["position_group"].value_counts())


Players:  521
position_group
Defender      191
Forward       156
Midfielder    134
Goalkeeper     40
Name: count, dtype: int64


In [35]:
METRICS = ["passes_per90", "progressive_passes_per90", "pass_accuracy_pct", "duels_won_pct", 
           "interceptions_per90", "recoveries_per90"]

def get_percentiles(player_name):       #Numbers per metric
    """
    Percentile of one player vs. the players of his own position group.
    """
    player = players_df[players_df["player_name"] == player_name].iloc[0]
    group = player["position_group"]
    same_group = players_df[players_df["position_group"] == group]

    percentiles = {}
    for metric in METRICS:
        # league_values = same_group[metric].dropna()
        league_values = pd.to_numeric(same_group[metric], errors='coerce').dropna()
        # player_value = player[metric]
        player_value = pd.to_numeric(pd.Series([player[metric]]), errors='coerce').iloc[0]

        if pd.isna(player_value) or league_values.empty:
            percentiles[metric] = None
        else:
            percentiles[metric] = round( 
                stats.percentileofscore(league_values, player_value, kind='rank')
            ) 
        # if pd.isna(player_name):
            # percentiles[metric] = None
        # else:
            # percentiles[metric] = round(
                # stats.percentileofscore(league_values, player_value, kind = "rank"),
            # )
    return group, percentiles

In [36]:
players_df.sort_values("minutes", ascending=False) [
    ["player_name", "position_group", "matches", "minutes"]
].head(10)

,player_name,position_group,matches,minutes
61,Jordan Pickford,Goalkeeper,7,720
226,Luka Modrić,Midfielder,7,720
51,Harry Maguire,Defender,7,675
34,John Stones,Defender,7,672
56,Danijel Subašić,Goalkeeper,6,663
62,Dejan Lovren,Defender,6,663
232,Ivan Rakitić,Midfielder,6,663
230,Domagoj Vida,Defender,6,663
246,Raphaël Varane,Defender,7,662
69,Thibaut Courtois,Goalkeeper,7,659


In [37]:
group, percentiles = get_percentiles("Kevin De Bruyne")
print("Compared against", group)
print(percentiles)

Compared against Forward
{'passes_per90': 96, 'progressive_passes_per90': 99, 'pass_accuracy_pct': 71, 'duels_won_pct': 96, 'interceptions_per90': 60, 'recoveries_per90': 84}


In [38]:
## SANITY CHECK
# Take the midfielder with the highest progressive_passes_per90 and check that his percentile 
# on that metric is close to 100:

players_df[players_df["position_group"] == "Midfielder"].sort_values(
    "progressive_passes_per90", ascending=False
)[["player_name", "progressive_passes_per90"]].head()

,player_name,progressive_passes_per90
326,Toni Kroos,37.993080
66,Granit Xhaka,21.910995
264,Javier Alejandro Mascherano,21.666667
262,Éver Maximiliano David Banega,20.952381
168,Andrés Iniesta Luján,20.510204


In [39]:
Kroos_Percentiles = get_percentiles("Toni Kroos")
print("Toni Kroos Metrics: ", Kroos_Percentiles)

Toni Kroos Metrics:  ('Midfielder', {'passes_per90': 99, 'progressive_passes_per90': 100, 'pass_accuracy_pct': 85, 'duels_won_pct': 76, 'interceptions_per90': 27, 'recoveries_per90': 93})


In [40]:
# Readable names for metrics (They will be the lables on dashboard)
LABLES = {
    "passes_per90": "Passes per 90",
    "progressive_passes_per90": "Progressive Passes per 90",
    "pass_accuracy_pct": "Pass Accuracy",
    "duels_won_pct": "Duels Won %",
    "interceptions_per90": "Interceptions per 90",
    "recoveries_per90": "Recoveries per 90",
}

PASSING = ["passes_per90", "progressive_passes_per90", "pass_accuracy_pct"]
DEFENDING = ["duels_won_pct", "interceptions_per90", "recoveries_per90"]

def average(percentiles, metrics):
    """
    Average percentile of a list of metrics (ignore empty values).
    """
    values = [percentiles[m]
            for m in metrics if percentiles[m] is not None]
            # The if ... is not None part matters: a player with no duels has no duels_won_pct, 
            # so his percentile is None, and adding None to numbers would crash. This filters those out.
    return sum(values) / len(values) if values else 0

def classify_archetype(percentiles):
    """
    Very simple archetype: is the pleyer better at defending or passing?
    """
    difference = average(percentiles, PASSING) - average(percentiles, DEFENDING)

    if difference > -15:
        return "Passing-Oriented"
    if difference < -15:
        return "Defensive-Oriented"
    return "Balanced"

def generate_scouting_report(percentiles):
    """
    Logic: >= 80 is a strength, <= 30 is a weakness.
    """
    strengths = []
    weaknesses = []

    for metric, value in percentiles.items():   # .item() gives each pair one a a time
        if value is None:
            continue
        if value >= 80:
            strengths.append(LABLES[metric])
        if value <= 30:
            weaknesses.append(LABLES[metric])

    return {
        "strengths": strengths,
        "weaknesses": weaknesses,
        "archetype": classify_archetype(percentiles),
    }

In [41]:
group, pct = get_percentiles("Toni Kroos")
report = generate_scouting_report(pct)

print("Compared against: ", group)
print("Strengths: ", report["strengths"])
print("Weaknesses", report["weaknesses"])
print("Archetype: ", report["archetype"])

Compared against:  Midfielder
Strengths:  ['Passes per 90', 'Progressive Passes per 90', 'Pass Accuracy', 'Recoveries per 90']
Weaknesses ['Interceptions per 90']
Archetype:  Passing-Oriented


In [42]:
# Import nedded libraries
import sys
sys.path.append('../src')
from models.rating import calculate_composite_rating

group, pct = get_percentiles("Toni Kroos")
print("Rating: ", calculate_composite_rating(pct, group))

Rating:  8.05


In [43]:
MIN_MINUTES_FOR_RATING = 180   # about 2 full matches; change it if you want

ratings = []

for _, row in players_df.iterrows():
    # Not enough minutes: no rating (the sample is too small to trust)
    if row["minutes"] < MIN_MINUTES_FOR_RATING:
        ratings.append(None)
        continue

    group, pct = get_percentiles(row["player_name"])
    ratings.append(calculate_composite_rating(pct, group))

players_df["rating"] = pd.Series(ratings, dtype="float")

print("Players with rating:", players_df["rating"].notna().sum())
print("Players without rating:", players_df["rating"].isna().sum())

Players with rating: 307
Players without rating: 214


In [44]:
# Check for top Midfielders
players_df[players_df["position_group"] == "Midfielder"].sort_values(
    "rating", ascending=False
)[["player_name", "matches", "minutes", "rating"]].head(10)

,player_name,matches,minutes,rating
262,Éver Maximiliano David Banega,2,189,8.18
130,Abdullah Ibrahim Otayf,3,252,8.05
326,Toni Kroos,3,289,8.05
106,N'Golo Kanté,6,567,7.74
134,Salman Mohammed Al Faraj,3,283,7.53
142,Salem Mohammed Al Dawsari,3,283,7.52
264,Javier Alejandro Mascherano,4,378,7.43
154,Jorge Resurrección Merodio,2,219,7.42
277,Lasse Schöne,3,236,7.25
66,Granit Xhaka,4,382,7.20


In [45]:
# Check bottom list
players_df[players_df["position_group"] == "Midfielder"].sort_values(
    "rating", ascending=True
)[["player_name", "matches", "minutes", "rating"]].head(10)

,player_name,matches,minutes,rating
243,Mathew Leckie,3,282,1.32
254,Robbie Kruse,3,207,1.58
334,Celso Borges Mora,3,290,2.74
396,Fakhreddine Ben Youssef,3,284,2.77
267,Birkir Bjarnason,3,280,2.84
190,Vahid Amiri,3,278,2.85
371,Emil Peter Forsberg,5,429,3.04
424,Carlos Alberto Sánchez Moreno,3,266,3.08
282,Édison Michael Flores Peralta,3,249,3.11
86,André Martín Carrillo Díaz,3,266,3.36


In [46]:
players_df["rating"].describe()

count    307.000000
mean       5.318990
std        1.577751
min        1.180000
25%        4.220000
50%        5.320000
75%        6.465000
max        9.410000
Name: rating, dtype: float64

In [47]:
# CHECK EVERYTHING WORKS
names = "Griezmann|Kruse|Leckie|Golovin|Amri|Iniesta|Modri"
check = league_df[league_df["player_name"].str.contains(names, case=False)]
check.groupby(["player_name", "position"]).size()

player_name           position                 
Aleksandr Golovin     Center Attacking Midfield    3
                      Right Center Midfield        1
Andrés Iniesta Luján  Center Attacking Midfield    1
                      Left Center Midfield         2
                      Left Midfield                1
Antoine Griezmann     Center Attacking Midfield    4
                      Center Forward               1
                      Right Center Forward         1
                      Right Wing Back              1
Luka Modrić           Right Attacking Midfield     1
                      Right Center Midfield        5
                      Right Defensive Midfield     1
Mathew Leckie         Right Midfield               2
                      Right Wing                   1
Robbie Kruse          Left Midfield                2
                      Left Wing                    1
dtype: int64

In [48]:
# CHECKIMG
league_df[league_df["position_group"] == "Midfielder"]["position"].value_counts()

position
Left Center Midfield         123
Right Center Midfield        120
Left Midfield                 76
Right Midfield                72
Center Defensive Midfield     51
Right Defensive Midfield      36
Left Defensive Midfield       36
Center Midfield                7
Name: count, dtype: int64

In [49]:
# Percentiles for any list of metrics       # NO ENTENDI

def get_metric_percentiles(player_name, metrics):
    """
    Percentile of onne player vs his position group, for any list of metrics.
    """
    matches = players_df[players_df["player_name"] == player_name]
    if matches.empty:
        raise ValueError(f"Player '{player_name}' not found. Search with str.contains().")
    player = matches.iloc[0]    # HUH?
    group = player["position_group"]
    same_group = players_df[players_df["position_group"] == group]

    percentiles = {}
    
    for metric in metrics:
        league_values = same_group[metric].dropna()
        player_value = player[metric]
        if pd.isna(player_value):
            percentiles[metric] = None
        else:
            percentiles[metric] = round(
                stats.percentileofscore(league_values, player_value, kind="rank"), 1
            )

    return group, percentiles

In [50]:
# Axes and Radar Value -> uses average func # HUH?

RADAR_AXES = {
    "PASS": ["pass_accuracy_pct", "passes_per90"],
    "DEFENSE": ["duels_won_pct", "interceptions_per90", "tackles_won_per90"],
    "SHOT": ["shots_per90", "xg_per90"],
    "DRIBBLE": ["dribbles_per90", "dribble_success_pct"],
    "PROGRESSION": ["progressive_passes_per90"],
    "RECOVERY": ["recoveries_per90"]
}

# All the metrics the radar needs, without repeat
RADAR_METRICS = sorted({m for metrics in RADAR_AXES.values() for m in metrics})

def get_radar_values(player_name):
    """
    Value (0-100) for each radar axis: the avarage of its percentiles.
    """
    group, pct = get_metric_percentiles(player_name, RADAR_METRICS)
    values = {axis: round(average(pct, metrics), 1) for axis, metrics in RADAR_AXES.items()}
    return group,values

In [53]:
# Import needed libraries
import plotly.graph_objects as go

def create_radar_chart(player_name):
    group, values = get_radar_values(player_name)
    categories = list(values.keys())
    player_vals = list(values.values())
    league_avg = [50] * len(categories) # the avg percentile is 50 by definition

    fig = go.Figure()

    # Player (the first point is repeated at the end to close the polygon)
    fig.add_trace(go.Scatterpolar(
        r  = player_vals + [player_vals[0]],
        theta = categories + [categories[0]],
        fill = "toself",
        fillcolor = "rgba(200, 255, 0, 0.3)",
        line = dict(color="#c8ff00", width=2),
        name = player_name,
    ))

    # League avg of his position group
    fig.add_trace(go.Scatterpolar(
        r = league_avg + [league_avg[0]],
        theta = categories + [categories[0]],
        fill = "toself",
        fillcolor = "rgba(0, 212, 255, 0.1)",
        line = dict(color="#00d4ff", width=2, dash="dot"),
        name = f"Average {group}",
    ))

    fig.update_layout(
        polar = dict(
            radialaxis = dict(visible=True, range=[0,100], gridcolor="#333", tickfont={"color": "#888"}),
            angularaxis = dict(tickfont={"color": "#e0e0e0", "size": 11}, gridcolor="#333"),
            bgcolor = "#0d1117",
        ),
        showlegend = True,
        legend = dict(font={"color": "#e0e0e0"}, bgcolor="rgba(0,0,0,0)"),
        paper_bgcolor = "#0a0e1a",
        font = dict(color="#e0e0e0"),
        title = f"{player_name}: attributes vs. {group} average",
        height = 450,
        margin = dict(l=60, r=60, t=70, b=40),
    )

    return fig

In [60]:
print(get_radar_values("Toni Kroos"))
create_radar_chart("Toni Kroos").show()

('Midfielder', {'PASS': np.float64(92.2), 'DEFENSE': np.float64(56.6), 'SHOT': np.float64(95.2), 'DRIBBLE': np.float64(75.2), 'PROGRESSION': np.float64(100.0), 'RECOVERY': np.float64(93.3)})


In [63]:
print(get_radar_values("Antoine Griezmann"))
create_radar_chart("Antoine Griezmann").show()

('Forward', {'PASS': np.float64(59.6), 'DEFENSE': np.float64(70.7), 'SHOT': np.float64(91.4), 'DRIBBLE': np.float64(57.8), 'PROGRESSION': np.float64(66.7), 'RECOVERY': np.float64(64.7)})


In [55]:
players_df[players_df["position_group"] == "Midfielder"].sort_values(
    "shots_per90", ascending=False
)[["player_name", "matches", "minutes", "shots", "shots_per90"]].head(10)

,player_name,matches,minutes,shots,shots_per90
326,Toni Kroos,3,289,16,4.982699
67,Philippe Coutinho Correia,5,453,23,4.569536
89,Alex Iwobi,1,61,3,4.426230
467,Takashi Usami,1,64,3,4.218750
268,Lucas Rodrigo Biglia,1,53,2,3.396226
371,Emil Peter Forsberg,5,429,15,3.146853
76,Naïm Sliti,2,148,5,3.040541
378,Thorgan Hazard,1,93,3,2.903226
243,Mathew Leckie,3,282,9,2.872340
364,Woo-Young Jung,1,99,3,2.727273
